In [ ]:
# Install required packages
%pip install -q transformers==4.35.0 torch==2.2.0 accelerate==0.24.0 bitsandbytes==0.41.1 huggingface_hub

# Import required packages
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import os

# Check GPU availability
print("🔍 Checking GPU...")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU found! Please make sure you've selected GPU in Runtime > Change runtime type")


In [ ]:
# Handle HuggingFace authentication
print("\n🔐 Setting up HuggingFace authentication...")

# First try to get token from environment variable
hf_token = os.getenv('HF_TOKEN')

if not hf_token:
    print("Please enter your HuggingFace token:")
    print("(You can find it at: https://huggingface.co/settings/tokens)")
    hf_token = input("Token: ").strip()

# Login to HuggingFace
login(token=hf_token)
print("✅ Successfully logged in to HuggingFace!")


In [ ]:
# Load base model
print("🚀 Loading CodeLlama-7b-Instruct...")

model_name = "codellama/CodeLlama-7b-Instruct-hf"

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True
)

# Load model with 8-bit quantization
print("Loading model in 8-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    device_map="auto",
    load_in_8bit=True,  # 8-bit quantization works well on Colab GPUs
    torch_dtype=torch.float16,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✅ Model loaded successfully!")

# Print model info
print("\n📊 Model Information:")
print(f"Model name: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
# Define generation function
def generate_response(prompt, max_new_tokens=512):
    """Generate response from the model."""
    formatted_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt", padding=True)
    
    # Move inputs to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(formatted_prompt):].strip()

# Test with a simple FastAPI prompt
test_prompt = "Create a FastAPI GET endpoint that returns 'Hello, World!'"
print("Testing basic inference...")
print("Prompt:", test_prompt)
print("\nGenerated Response:")
print("-" * 40)
print(generate_response(test_prompt))
print("-" * 40)


In [ ]:
# Test more complex FastAPI prompts
complex_prompts = [
    "Create a FastAPI endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting",
    "Create a FastAPI endpoint that handles file upload and saves the file to disk",
    "Create a FastAPI endpoint that implements basic authentication using JWT tokens"
]

print("Testing complex prompts...")
for i, prompt in enumerate(complex_prompts, 1):
    print(f"\nTest {i}:")
    print("Prompt:", prompt)
    print("\nGenerated Response:")
    print("-" * 40)
    print(generate_response(prompt))
    print("-" * 40)


In [ ]:
# FastAPI Evaluator Implementation
import ast
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import json

def extract_code_from_markdown(text: str) -> str:
    """Extract Python code from markdown-formatted text with code blocks."""
    # Find content between triple backticks
    code_blocks = re.findall(r'```(?:python)?(.*?)```', text, re.DOTALL)
    if code_blocks:
        # Return the first code block found
        return code_blocks[0].strip()
    # If no code blocks found, try to find the code directly
    # Remove any explanatory text that comes before or after the code
    if 'from fastapi import' in text:
        # Find the first import statement and everything after it
        code = text[text.find('from fastapi import'):]
        # Remove any explanatory text that might come after the code
        if 'This endpoint' in code:
            code = code[:code.find('This endpoint')]
        return code.strip()
    return text.strip()

@dataclass
class EvaluationResult:
    """Stores the evaluation results for a single test case."""
    prompt: str
    response: str
    has_imports: bool = False
    has_router: bool = False
    has_endpoint: bool = False
    has_type_hints: bool = False
    has_docstring: bool = False
    has_error_handling: bool = False
    is_valid_python: bool = False
    extracted_endpoints: List[Dict[str, Any]] = None
    score: float = 0.0
    error_message: Optional[str] = None

class FastAPIEvaluator:
    """Evaluates FastAPI code generation responses."""
    
    def __init__(self):
        self.required_imports = [
            "fastapi",
            "FastAPI",
            "APIRouter",
            "HTTPException",
            "status",
            "Response",
            "Request"
        ]
    
    def evaluate_response(self, prompt: str, response: str) -> EvaluationResult:
        """Evaluates a single response."""
        # First extract actual code from the response
        code = extract_code_from_markdown(response)
        result = EvaluationResult(prompt=prompt, response=code)
        
        try:
            # Check if it's valid Python code
            ast.parse(code)
            result.is_valid_python = True
        except SyntaxError as e:
            result.error_message = f"Invalid Python syntax: {str(e)}"
            return result
        
        # Check for imports
        result.has_imports = any(imp in response for imp in self.required_imports)
        
        # Check for router/app initialization
        result.has_router = bool(re.search(r"(app\s*=\s*FastAPI\(\)|router\s*=\s*APIRouter\(\))", response))
        
        # Check for endpoints
        endpoint_pattern = r"@\s*(app|router)\.(get|post|put|delete|patch)\s*\(\s*['\"]([^'\"]+)['\"]\s*\)"
        endpoints = re.finditer(endpoint_pattern, response)
        result.extracted_endpoints = []
        
        for match in endpoints:
            decorator, method, path = match.groups()
            result.extracted_endpoints.append({
                "decorator": decorator,
                "method": method.upper(),
                "path": path
            })
        
        result.has_endpoint = len(result.extracted_endpoints) > 0
        
        # Check for type hints
        result.has_type_hints = bool(re.search(r"def\s+\w+\s*\([^)]*:\s*\w+(\s*=\s*[^,)]+)?", response))
        
        # Check for docstrings
        result.has_docstring = '"""' in response or "'''" in response
        
        # Check for error handling
        result.has_error_handling = "HTTPException" in response or "try:" in response
        
        # Calculate score (simple scoring system)
        score_components = [
            result.is_valid_python,
            result.has_imports,
            result.has_router,
            result.has_endpoint,
            result.has_type_hints,
            result.has_docstring,
            result.has_error_handling
        ]
        result.score = sum(1 for component in score_components if component) / len(score_components)
        
        return result
    
    def format_evaluation_result(self, result: EvaluationResult) -> str:
        """Formats the evaluation result into a readable string."""
        output = []
        output.append("📊 Evaluation Results:")
        output.append(f"✓ Valid Python: {result.is_valid_python}")
        if result.error_message:
            output.append(f"⚠️ Error: {result.error_message}")
        output.append(f"✓ Has Imports: {result.has_imports}")
        output.append(f"✓ Has Router/App: {result.has_router}")
        output.append(f"✓ Has Endpoints: {result.has_endpoint}")
        if result.extracted_endpoints:
            output.append("\nEndpoints found:")
            for endpoint in result.extracted_endpoints:
                output.append(f"  • {endpoint['method']} {endpoint['path']}")
        output.append(f"\n✓ Has Type Hints: {result.has_type_hints}")
        output.append(f"✓ Has Docstrings: {result.has_docstring}")
        output.append(f"✓ Has Error Handling: {result.has_error_handling}")
        output.append(f"\n🎯 Overall Score: {result.score:.2%}")
        return "\n".join(output)

# Initialize evaluator
evaluator = FastAPIEvaluator()


In [ ]:
# Test cases for evaluation
test_cases = [
    {
        "name": "Dataset Example - Project List Endpoint",
        "prompt": "Create a FastAPI GET endpoint that lists projects for an organization. It should: 1) Accept organization_id as a parameter, 2) Use database session from dependencies, 3) Return a list of projects, 4) Include proper error handling if no projects found. Return ONLY the code without any explanation."
    },
    {
        "name": "Simple Hello World",
        "prompt": "Create a FastAPI GET endpoint that returns 'Hello, World!'. Return ONLY the code without any explanation."
    },
    {
        "name": "JSON Payload Handler",
        "prompt": "Create a FastAPI POST endpoint that accepts a JSON payload with 'name' and 'age' fields and returns a greeting. Return ONLY the code without any explanation."
    },
    {
        "name": "Error Handling",
        "prompt": "Create a FastAPI endpoint that demonstrates proper error handling with HTTPException. Return ONLY the code without any explanation."
    },
    {
        "name": "Path Parameters",
        "prompt": "Create a FastAPI endpoint that accepts a user_id as a path parameter and returns user information. Return ONLY the code without any explanation."
    }
]

# Run evaluations
print("🧪 Running FastAPI Code Generation Tests\n")

all_scores = []
for test in test_cases:
    print(f"\n{'='*60}")
    print(f"Test: {test['name']}")
    print(f"Prompt: {test['prompt']}")
    print(f"{'='*60}\n")
    
    # Generate response
    response = generate_response(test['prompt'])
    print("Generated Code:")
    print("-" * 40)
    print(response)
    print("-" * 40)
    
    # Evaluate response
    result = evaluator.evaluate_response(test['prompt'], response)
    print("\nEvaluation:")
    print(evaluator.format_evaluation_result(result))
    all_scores.append(result.score)

# Print summary
print("\n📈 Overall Results:")
print(f"Average Score: {sum(all_scores)/len(all_scores):.2%}")
print(f"Best Score: {max(all_scores):.2%}")
print(f"Worst Score: {min(all_scores):.2%}")
